# 1. Project Introduction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SayedaHaneen/data-augmentation-imagedatagenerator/blob/main/Data_Augmentation_ImageDataGenerator.ipynb)

**Course Project:** BS Computer Science — Deep Learning  
**Student:** Sayeda Haneen  
**Topic:** Data Augmentation with ImageDataGenerator: Fighting Overfitting on Small Image Datasets  
**Theoretical Reference:** François Chollet, *Deep Learning with Python* (Chapter 5: Deep Learning for Computer Vision)

---

### Project Overview
Training deep Convolutional Neural Networks (CNNs) on small computer vision datasets is one of the most common challenges in applied deep learning. When a dataset has only a few thousand samples, a deep model with large representational capacity can rapidly **memorize** training images rather than learning robust, invariant features (such as edges, shapes, textures, and anatomical structures). This phenomenon is known as **overfitting**.

The symptoms of overfitting:
1. **Training accuracy** continues to climb steadily towards 95%–100%.
2. **Validation accuracy** stagnates early (often around 70%–75%) or begins to drop.
3. **Validation loss** stops decreasing and starts climbing, indicating that the model's confidence on unseen data is degrading.

### Objective
This project demonstrates how **real-time data augmentation** using TensorFlow / Keras `ImageDataGenerator` mitigates overfitting.

We construct a controlled, reproducible scientific experiment:
* **Identical CNN architecture:** 4 convolutional blocks + dense classification head.
* **Identical datasets:** 2,000 balanced training images (1,000 cats, 1,000 dogs) and 1,000 validation images (500 cats, 500 dogs) from the Microsoft Kaggle Cats vs Dogs dataset.
* **Identical training parameters:** 15 epochs, batch size of 32, RMSprop optimizer ($lr = 10^{-4}$), binary crossentropy loss.
* **Controlled variable:**
  * **Experiment 1:** Without Data Augmentation (`rescale=1./255` only).
  * **Experiment 2:** With Data Augmentation (`rescale=1./255`, `rotation_range=40`, `width_shift_range=0.2`, `zoom_range=0.2`, `horizontal_flip=True`).
  * **Validation Rule:** Validation data is **never augmented** in either experiment.


## 2. Import Libraries

In this section, we import all required Python libraries:
* **TensorFlow / Keras:** For building CNN models, defining layers, optimizers, and data generators.
* **PIL (Pillow):** For image file verification and format conversion.
* **NumPy & Pandas:** For numerical operations and structured comparison tables.
* **Matplotlib:** For visualizing dataset samples, augmentation examples, and side-by-side training curves.
* **Standard Library (`os`, `shutil`, `zipfile`, `urllib`):** For file management, dataset download, and extraction.

We also verify whether a GPU is available and set fixed random seeds (`SEED = 42`) across Python, NumPy, and TensorFlow to ensure reproducible dataset splits and weight initializations.


In [ ]:
import os
import shutil
import random
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f"TensorFlow Version: {tf.__version__}")

# Hardware acceleration check (GPU vs CPU)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Detected: {gpus}")
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except RuntimeError as e:
            print(f"GPU config notice: {e}")
else:
    print("⚠️ Warning: No GPU detected. TensorFlow will run on CPU.")
    print("   If you are running in Google Colab, enable GPU acceleration via:")
    print("   Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")

# Set random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print(f"Reproducibility seed set to: {SEED}")


## 3. Download Dataset

We download the official Microsoft Kaggle Cats and Dogs dataset archive (`kagglecatsanddogs_5340.zip`, ~786 MB) directly from the official Microsoft download server.

The download script includes a progress reporter and automatically checks if the archive already exists on disk to prevent redundant downloads if the notebook is re-run.


In [ ]:
DATASET_URL = "https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip"
ZIP_FILE_PATH = "kagglecatsanddogs_5340.zip"
RAW_DATASET_DIR = "dataset_raw"

def download_dataset(url, dest_path):
    if os.path.exists(dest_path):
        size_mb = os.path.getsize(dest_path) / (1024 * 1024)
        print(f"Archive '{dest_path}' already exists ({size_mb:.1f} MB). Skipping download.")
        return
    
    print(f"Downloading Kaggle Cats vs Dogs dataset from:\n{url}")
    print("File size is ~786 MB. Please wait a moment...")

    class DownloadProgress:
        def __init__(self):
            self.last_percent = -1
        def __call__(self, block_num, block_size, total_size):
            downloaded = block_num * block_size
            if total_size > 0:
                percent = int(downloaded * 100 / total_size)
                if percent != self.last_percent and percent % 10 == 0:
                    mb_down = downloaded / (1024 * 1024)
                    mb_tot = total_size / (1024 * 1024)
                    print(f"  Downloaded: {percent}% ({mb_down:.1f} / {mb_tot:.1f} MB)")
                    self.last_percent = percent

    urllib.request.urlretrieve(url, dest_path, reporthook=DownloadProgress())
    print("Dataset download complete!")

download_dataset(DATASET_URL, ZIP_FILE_PATH)


## 4. Extract Dataset

We extract the downloaded ZIP file into `dataset_raw/`. The extracted directory structure contains:
```
PetImages/
    Cat/   (12,500 images)
    Dog/   (12,500 images)
```


In [ ]:
def extract_dataset(zip_path, extract_dir):
    petimages_dir = os.path.join(extract_dir, "PetImages")
    if os.path.exists(petimages_dir):
        print(f"Dataset already extracted at '{petimages_dir}'. Skipping extraction.")
        return petimages_dir
    
    print(f"Extracting '{zip_path}' to '{extract_dir}'...")
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction completed successfully!")
    return petimages_dir

extracted_petimages_path = extract_dataset(ZIP_FILE_PATH, RAW_DATASET_DIR)


## 5. Remove Corrupted Images

The original Microsoft Kaggle Cats vs Dogs dataset is famous for containing a few invalid, zero-byte, or malformed image files (such as `Cat/666.jpg` and `Dog/11702.jpg`), along with non-image OS metadata files like `Thumbs.db`.

In this step:
1. We iterate over every file in the `Cat/` and `Dog/` folders.
2. We filter only `.jpg` / `.jpeg` files.
3. We run PIL's `Image.verify()` to validate the JPEG header and structure.
4. We attempt to open and load the pixel data to guarantee the file is completely uncorrupted.
5. Corrupted images are counted and safely skipped, preventing runtime crashes during training.


In [ ]:
def filter_corrupted_images(folder_path):
    """
    Scans an image directory, validates each image using PIL Image.verify()
    and Image.load(), and returns a list of verified valid image paths.
    """
    valid_images = []
    corrupted_count = 0
    
    filenames = sorted(os.listdir(folder_path))
    for fname in filenames:
        if not fname.lower().endswith(('.jpg', '.jpeg')):
            continue
        fpath = os.path.join(folder_path, fname)
        try:
            with Image.open(fpath) as img:
                img.verify()
            # Double check that image data decodes completely
            with Image.open(fpath) as img:
                img.load()
            valid_images.append(fpath)
        except Exception:
            corrupted_count += 1

    return valid_images, corrupted_count

cat_raw_folder = os.path.join(extracted_petimages_path, "Cat")
dog_raw_folder = os.path.join(extracted_petimages_path, "Dog")

print("Scanning and verifying images with PIL Image.verify()...")
valid_cat_paths, corrupt_cats = filter_corrupted_images(cat_raw_folder)
valid_dog_paths, corrupt_dogs = filter_corrupted_images(dog_raw_folder)

print(f"Cat Directory -> Valid Images: {len(valid_cat_paths):,d} | Corrupted/Skipped: {corrupt_cats}")
print(f"Dog Directory -> Valid Images: {len(valid_dog_paths):,d} | Corrupted/Skipped: {corrupt_dogs}")


## 6. Create Balanced Training and Validation Sets

To reproduce the small-dataset scenario described in François Chollet's *Deep Learning with Python*, we construct a balanced, disjoint dataset:
* **Training Set:**
  * 1,000 Cat images
  * 1,000 Dog images
  * **Total: 2,000 training images**
* **Validation Set:**
  * 500 Cat images
  * 500 Dog images
  * **Total: 1,000 validation images**

### Strict Data Integrity Rules
* **Reproducibility:** Shuffled using a fixed random seed (`SEED = 42`).
* **Zero Data Leakage:** We strictly verify that training and validation sets are mathematically disjoint (`set(train).isdisjoint(set(val))`). No image ever appears in both sets.


In [ ]:
SUBSET_BASE_DIR = "cats_and_dogs_small"
TRAIN_DIR = os.path.join(SUBSET_BASE_DIR, "train")
VAL_DIR = os.path.join(SUBSET_BASE_DIR, "validation")

TRAIN_CATS_DIR = os.path.join(TRAIN_DIR, "cats")
TRAIN_DOGS_DIR = os.path.join(TRAIN_DIR, "dogs")
VAL_CATS_DIR = os.path.join(VAL_DIR, "cats")
VAL_DOGS_DIR = os.path.join(VAL_DIR, "dogs")

for target_dir in [TRAIN_CATS_DIR, TRAIN_DOGS_DIR, VAL_CATS_DIR, VAL_DOGS_DIR]:
    os.makedirs(target_dir, exist_ok=True)

# Shuffle deterministically using SEED
random.seed(SEED)
shuffled_cat_list = list(valid_cat_paths)
shuffled_dog_list = list(valid_dog_paths)
random.shuffle(shuffled_cat_list)
random.shuffle(shuffled_dog_list)

# Split into balanced subsets
train_cat_subset = shuffled_cat_list[:1000]
val_cat_subset = shuffled_cat_list[1000:1500]

train_dog_subset = shuffled_dog_list[:1000]
val_dog_subset = shuffled_dog_list[1000:1500]

# Verify zero data leakage
assert set(train_cat_subset).isdisjoint(set(val_cat_subset)), "Data Leakage: Cat train/val sets overlap!"
assert set(train_dog_subset).isdisjoint(set(val_dog_subset)), "Data Leakage: Dog train/val sets overlap!"

def copy_images_to_directory(image_list, destination_folder):
    for src_path in image_list:
        fname = os.path.basename(src_path)
        dest_path = os.path.join(destination_folder, fname)
        if not os.path.exists(dest_path):
            shutil.copyfile(src_path, dest_path)

print("Copying selected images to small dataset directories...")
copy_images_to_directory(train_cat_subset, TRAIN_CATS_DIR)
copy_images_to_directory(val_cat_subset, VAL_CATS_DIR)
copy_images_to_directory(train_dog_subset, TRAIN_DOGS_DIR)
copy_images_to_directory(val_dog_subset, VAL_DOGS_DIR)

# Confirm counts
n_train_cats = len(os.listdir(TRAIN_CATS_DIR))
n_train_dogs = len(os.listdir(TRAIN_DOGS_DIR))
n_val_cats = len(os.listdir(VAL_CATS_DIR))
n_val_dogs = len(os.listdir(VAL_DOGS_DIR))

print("\n--- Dataset Verification Summary ---")
print(f"Training Cats:       {n_train_cats:4d} images")
print(f"Training Dogs:       {n_train_dogs:4d} images")
print(f"Total Training:      {n_train_cats + n_train_dogs:4d} images")
print(f"Validation Cats:     {n_val_cats:4d} images")
print(f"Validation Dogs:     {n_val_dogs:4d} images")
print(f"Total Validation:    {n_val_cats + n_val_dogs:4d} images")
print(f"Grand Total Used:    {n_train_cats + n_train_dogs + n_val_cats + n_val_dogs:4d} images")


## 7. Visualize Dataset

We display a sample grid of Cat and Dog images from our balanced training dataset. This visual check confirms correct class folder organization, shows the natural diversity of pet poses, lighting conditions, and aspect ratios, and demonstrates why standardized resizing ($150 \times 150$) is necessary.


In [ ]:
def plot_sample_images():
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    fig.suptitle("Sample Images from Small Balanced Cats vs Dogs Dataset", fontsize=16, fontweight='bold')

    sample_cats = [os.path.join(TRAIN_CATS_DIR, f) for f in sorted(os.listdir(TRAIN_CATS_DIR))[:4]]
    sample_dogs = [os.path.join(TRAIN_DOGS_DIR, f) for f in sorted(os.listdir(TRAIN_DOGS_DIR))[:4]]

    for idx, path in enumerate(sample_cats):
        img = Image.open(path).convert('RGB').resize((150, 150))
        axes[0, idx].imshow(img)
        axes[0, idx].set_title(f"Cat (Class 0)\n{os.path.basename(path)}")
        axes[0, idx].axis('off')

    for idx, path in enumerate(sample_dogs):
        img = Image.open(path).convert('RGB').resize((150, 150))
        axes[1, idx].imshow(img)
        axes[1, idx].set_title(f"Dog (Class 1)\n{os.path.basename(path)}")
        axes[1, idx].axis('off')

    plt.tight_layout()
    plt.show()

plot_sample_images()


## 8. Demonstrate Data Augmentation

### How ImageDataGenerator Works
Data augmentation generates believable synthetic training images by applying random geometric transformations to existing training samples:
* **Real-time execution:** Transformations are computed **in-memory** on CPU/GPU as each mini-batch is loaded.
* **Zero disk expansion:** The physical dataset on disk remains unchanged.
* **Epoch variety:** Because transformations are random, the model effectively never sees the exact same image twice across 15 epochs.

### Demonstration
Below, we select one cat image from our training set and generate **5 augmented variations** using the exact configuration required for Experiment 2:
* `rotation_range=40`: Random rotations up to 40 degrees.
* `width_shift_range=0.2`: Random horizontal translation (up to 20% of image width).
* `zoom_range=0.2`: Random zoom in / out by up to 20%.
* `horizontal_flip=True`: Random horizontal mirror reflections.


In [ ]:
demo_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Select one representative cat image
demo_cat_path = os.path.join(TRAIN_CATS_DIR, sorted(os.listdir(TRAIN_CATS_DIR))[0])
demo_img = Image.open(demo_cat_path).convert('RGB').resize((150, 150))
img_array = np.array(demo_img)
img_array = img_array.reshape((1,) + img_array.shape) # Shape: (1, 150, 150, 3)

fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
fig.suptitle("Data Augmentation Demonstration: 1 Original + 5 Augmented Variations", fontsize=14, fontweight='bold')

# Plot original image
axes[0].imshow(demo_img)
axes[0].set_title("Original Image", color='navy', fontweight='bold')
axes[0].axis('off')

# Plot 5 augmented variations
i = 1
for batch in demo_datagen.flow(img_array, batch_size=1, seed=SEED):
    axes[i].imshow(batch[0])
    axes[i].set_title(f"Augmented #{i}", color='darkgreen')
    axes[i].axis('off')
    i += 1
    if i > 5:
        break

plt.tight_layout()
plt.show()


## 9. Build CNN Architecture

We use the lightweight convolutional neural network architecture presented in François Chollet's *Deep Learning with Python* (Chapter 5):
1. **Four Convolutional Blocks:**
   * Block 1: `Conv2D(32, (3, 3), activation='relu')` + `MaxPooling2D((2, 2))`
   * Block 2: `Conv2D(64, (3, 3), activation='relu')` + `MaxPooling2D((2, 2))`
   * Block 3: `Conv2D(128, (3, 3), activation='relu')` + `MaxPooling2D((2, 2))`
   * Block 4: `Conv2D(128, (3, 3), activation='relu')` + `MaxPooling2D((2, 2))`
2. **Classifier Head:**
   * `Flatten()`: Unrolls 3D feature maps ($7 \times 7 \times 128 = 6,272$) into a 1D vector.
   * `Dense(512, activation='relu')`: Fully connected representation layer.
   * `Dense(1, activation='sigmoid')`: Single scalar output representing the probability of class 1 (Dog).
3. **Compilation:**
   * Loss: `binary_crossentropy`
   * Optimizer: `optimizers.RMSprop(learning_rate=1e-4)`
   * Metric: `['accuracy']`

We define a builder function `build_model()` with `tf.keras.utils.set_random_seed(SEED)` to guarantee that both experiments start from identical architectures and initial weights.


In [ ]:
def build_model():
    """
    Builds and compiles the lightweight Chollet CNN architecture.
    Uses a fixed random seed to ensure reproducible weight initialization.
    """
    tf.keras.utils.set_random_seed(SEED)
    
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
        layers.MaxPooling2D((2, 2)),
        
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Block 4
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        # Classifier
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        loss='binary_crossentropy',
        optimizer=optimizers.RMSprop(learning_rate=1e-4),
        metrics=['accuracy']
    )
    return model

# Inspect the CNN architecture summary
baseline_cnn = build_model()
baseline_cnn.summary()


## 10. Experiment 1: Without Augmentation

In this baseline experiment:
* **Training Data Generator:** `ImageDataGenerator(rescale=1./255)` (normalizing pixel values to $[0, 1]$).
* **Validation Data Generator:** `ImageDataGenerator(rescale=1./255)`.
* **Image dimensions:** $150 \times 150$.
* **Batch size:** 32.
* **Epochs:** Exactly 15.

We instantiate a fresh model instance (`model_no_aug`) and record the full training history (`history_no_aug`).


In [ ]:
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 15

print("Configuring Data Generators for Experiment 1 (Without Augmentation)...")

train_datagen_no_aug = ImageDataGenerator(rescale=1./255)
val_datagen_no_aug = ImageDataGenerator(rescale=1./255)

train_generator_no_aug = train_datagen_no_aug.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator_no_aug = val_datagen_no_aug.flow_from_directory(
    VAL_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# Build fresh model instance
model_no_aug = build_model()

print(f"\n--- Training Experiment 1: Baseline CNN ({EPOCHS} Epochs) ---")
history_no_aug = model_no_aug.fit(
    train_generator_no_aug,
    steps_per_epoch=len(train_generator_no_aug),
    epochs=EPOCHS,
    validation_data=val_generator_no_aug,
    validation_steps=len(val_generator_no_aug),
    verbose=1
)
print("Experiment 1 Training Completed Successfully!")


## 11. Experiment 2: With Augmentation

In this experiment:
* **Training Data Generator:** Configured with the exact required parameters:
  `ImageDataGenerator(rescale=1./255, rotation_range=40, width_shift_range=0.2, zoom_range=0.2, horizontal_flip=True)`
* **Validation Data Generator:** Configured with `ImageDataGenerator(rescale=1./255)` only. **Validation images are never augmented.**
* **Image dimensions:** $150 \times 150$.
* **Batch size:** 32.
* **Epochs:** Exactly 15.

We instantiate an independent model instance (`model_aug`) with the identical architecture and initial seed, and train for 15 epochs, saving `history_aug`.


In [ ]:
print("Configuring Data Generators for Experiment 2 (With Augmentation)...")

train_datagen_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Golden Rule: Never augment validation data
val_datagen_aug = ImageDataGenerator(rescale=1./255)

train_generator_aug = train_datagen_aug.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator_aug = val_datagen_aug.flow_from_directory(
    VAL_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# Build independent model instance with identical architecture
model_aug = build_model()

print(f"\n--- Training Experiment 2: Augmented CNN ({EPOCHS} Epochs) ---")
history_aug = model_aug.fit(
    train_generator_aug,
    steps_per_epoch=len(train_generator_aug),
    epochs=EPOCHS,
    validation_data=val_generator_aug,
    validation_steps=len(val_generator_aug),
    verbose=1
)
print("Experiment 2 Training Completed Successfully!")


## 12. Compare Training Results

We extract the actual training metrics directly from `history_no_aug.history` and `history_aug.history` to build a clean comparison table.

*Important:* No values are hardcoded or fabricated — all metrics are derived dynamically from actual training histories.


In [ ]:
# Dynamically extract actual performance metrics
exp1_train_acc = history_no_aug.history['accuracy'][-1]
exp1_val_acc = history_no_aug.history['val_accuracy'][-1]
exp1_train_loss = history_no_aug.history['loss'][-1]
exp1_val_loss = history_no_aug.history['val_loss'][-1]
exp1_peak_val_acc = max(history_no_aug.history['val_accuracy'])
exp1_min_val_loss = min(history_no_aug.history['val_loss'])

exp2_train_acc = history_aug.history['accuracy'][-1]
exp2_val_acc = history_aug.history['val_accuracy'][-1]
exp2_train_loss = history_aug.history['loss'][-1]
exp2_val_loss = history_aug.history['val_loss'][-1]
exp2_peak_val_acc = max(history_aug.history['val_accuracy'])
exp2_min_val_loss = min(history_aug.history['val_loss'])

comparison_dict = {
    "Metric": [
        "Final Training Accuracy (Epoch 15)",
        "Final Validation Accuracy (Epoch 15)",
        "Final Training Loss (Epoch 15)",
        "Final Validation Loss (Epoch 15)",
        "Peak Validation Accuracy",
        "Lowest Validation Loss",
        "Final Generalization Gap (|Train - Val Acc|)"
    ],
    "Without Augmentation (Exp 1)": [
        f"{exp1_train_acc:.4f} ({exp1_train_acc*100:.2f}%)",
        f"{exp1_val_acc:.4f} ({exp1_val_acc*100:.2f}%)",
        f"{exp1_train_loss:.4f}",
        f"{exp1_val_loss:.4f}",
        f"{exp1_peak_val_acc:.4f} ({exp1_peak_val_acc*100:.2f}%)",
        f"{exp1_min_val_loss:.4f}",
        f"{abs(exp1_train_acc - exp1_val_acc)*100:.2f}%"
    ],
    "With Augmentation (Exp 2)": [
        f"{exp2_train_acc:.4f} ({exp2_train_acc*100:.2f}%)",
        f"{exp2_val_acc:.4f} ({exp2_val_acc*100:.2f}%)",
        f"{exp2_train_loss:.4f}",
        f"{exp2_val_loss:.4f}",
        f"{exp2_peak_val_acc:.4f} ({exp2_peak_val_acc*100:.2f}%)",
        f"{exp2_min_val_loss:.4f}",
        f"{abs(exp2_train_acc - exp2_val_acc)*100:.2f}%"
    ]
}

comparison_table = pd.DataFrame(comparison_dict)
print("=== EXPERIMENTAL TRAINING RESULTS COMPARISON ===")
display(comparison_table)


## 13. Plot Accuracy and Loss Curves

We generate side-by-side plots comparing both experiments:
* **Training curves:** plotted in **GREEN**.
* **Validation curves:** plotted in **RED**.
* **X-axis:** Epochs ($1$ to $15$).
* **Y-axis:** Metric value.

### Observations to Note
* In **Experiment 1 (Without Augmentation)**: Notice how training accuracy rapidly outpaces validation accuracy, while validation loss bottoms out early and begins to climb — classic signatures of overfitting.
* In **Experiment 2 (With Augmentation)**: Notice how training and validation trajectories track much closer together, demonstrating superior regularization.


In [ ]:
epochs_range = range(1, EPOCHS + 1)

# 1. Side-by-Side Accuracy Plots
fig, (ax_acc1, ax_acc2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Without Augmentation Accuracy
ax_acc1.plot(epochs_range, history_no_aug.history['accuracy'], color='green', marker='o', linewidth=2, label='Training Accuracy')
ax_acc1.plot(epochs_range, history_no_aug.history['val_accuracy'], color='red', marker='s', linewidth=2, label='Validation Accuracy')
ax_acc1.set_title('Experiment 1: Without Data Augmentation\n(Overfitting Evident)', fontsize=13, fontweight='bold')
ax_acc1.set_xlabel('Epochs', fontsize=11)
ax_acc1.set_ylabel('Accuracy', fontsize=11)
ax_acc1.set_ylim([0.4, 1.0])
ax_acc1.set_xticks(epochs_range)
ax_acc1.legend(loc='lower right')
ax_acc1.grid(True, linestyle='--', alpha=0.6)

# Plot 2: With Augmentation Accuracy
ax_acc2.plot(epochs_range, history_aug.history['accuracy'], color='green', marker='o', linewidth=2, label='Training Accuracy')
ax_acc2.plot(epochs_range, history_aug.history['val_accuracy'], color='red', marker='s', linewidth=2, label='Validation Accuracy')
ax_acc2.set_title('Experiment 2: With Data Augmentation\n(Improved Generalization)', fontsize=13, fontweight='bold')
ax_acc2.set_xlabel('Epochs', fontsize=11)
ax_acc2.set_ylabel('Accuracy', fontsize=11)
ax_acc2.set_ylim([0.4, 1.0])
ax_acc2.set_xticks(epochs_range)
ax_acc2.legend(loc='lower right')
ax_acc2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# 2. Side-by-Side Loss Plots
fig, (ax_loss1, ax_loss2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot 3: Without Augmentation Loss
ax_loss1.plot(epochs_range, history_no_aug.history['loss'], color='green', marker='o', linewidth=2, label='Training Loss')
ax_loss1.plot(epochs_range, history_no_aug.history['val_loss'], color='red', marker='s', linewidth=2, label='Validation Loss')
ax_loss1.set_title('Experiment 1 Loss: Without Augmentation\n(Validation Loss Stagnates / Diverges)', fontsize=13, fontweight='bold')
ax_loss1.set_xlabel('Epochs', fontsize=11)
ax_loss1.set_ylabel('Loss (Binary Crossentropy)', fontsize=11)
ax_loss1.set_xticks(epochs_range)
ax_loss1.legend(loc='upper right')
ax_loss1.grid(True, linestyle='--', alpha=0.6)

# Plot 4: With Augmentation Loss
ax_loss2.plot(epochs_range, history_aug.history['loss'], color='green', marker='o', linewidth=2, label='Training Loss')
ax_loss2.plot(epochs_range, history_aug.history['val_loss'], color='red', marker='s', linewidth=2, label='Validation Loss')
ax_loss2.set_title('Experiment 2 Loss: With Augmentation\n(Training and Validation Remain Coupled)', fontsize=13, fontweight='bold')
ax_loss2.set_xlabel('Epochs', fontsize=11)
ax_loss2.set_ylabel('Loss (Binary Crossentropy)', fontsize=11)
ax_loss2.set_xticks(epochs_range)
ax_loss2.legend(loc='upper right')
ax_loss2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


## 14. Understand Overfitting

### The Mechanics of Overfitting on Small Datasets
1. **High Model Capacity vs. Low Sample Variance:** A deep convolutional network possesses hundreds of thousands (or millions) of trainable parameters. With only 2,000 images, the network has enough degree-of-freedom capacity to memorize training samples pixel-by-pixel like an associative lookup table.
2. **Spurious Correlations:** Instead of learning invariant visual features (e.g., cat ear triangles, feline whisker patterns, canine snouts), the model learns spurious correlations (e.g., green grassy backgrounds, specific lighting artifacts, camera orientations).
3. **The Generalization Gap:** The divergence between training accuracy and validation accuracy represents the generalization gap. A large gap indicates that the model has failed to generalize to unseen test distributions.

### How Data Augmentation Fights Overfitting
* **Infinite Pseudo-Unique Samples:** By rotating, shifting, zooming, and flipping images, the network never encounters the identical tensor representation twice.
* **Forced Feature Invariance:** The model is prevented from memorizing specific pixel positions and is forced to learn spatial and geometric invariants.
* **Implicit Regularization:** Data augmentation functions as an implicit regularizer, narrowing the generalization gap without shrinking the network's expressive capacity.

*Disclaimer: The diagram below illustrates conceptual learning dynamics to visualize these theoretical principles.*


In [ ]:
# Conceptual comparison diagram of overfitting vs data augmentation regularization
sim_epochs = np.linspace(1, 15, 100)

fig, (diag1, diag2) = plt.subplots(1, 2, figsize=(15, 5))

# Conceptual Curve 1: Severe Overfitting
sim_train_no_aug = 0.52 + 0.44 * (1 - np.exp(-sim_epochs / 3.0))
sim_val_no_aug = 0.52 + 0.22 * (1 - np.exp(-sim_epochs / 2.2)) - 0.08 * (sim_epochs / 15.0)**1.6

diag1.plot(sim_epochs, sim_train_no_aug, 'g-', linewidth=2.5, label='Training Accuracy (Memorization)')
diag1.plot(sim_epochs, sim_val_no_aug, 'r--', linewidth=2.5, label='Validation Accuracy (Stagnation)')
diag1.fill_between(sim_epochs, sim_train_no_aug, sim_val_no_aug, color='salmon', alpha=0.25, label='Generalization Gap (Overfitting)')
diag1.set_title('Conceptual Model: Without Data Augmentation\n(Memorization of Limited Training Samples)', fontsize=12, fontweight='bold')
diag1.set_xlabel('Epochs', fontsize=11)
diag1.set_ylabel('Accuracy', fontsize=11)
diag1.set_ylim([0.4, 1.02])
diag1.legend(loc='lower right')
diag1.grid(True, linestyle=':', alpha=0.6)

# Conceptual Curve 2: Data Augmentation Regularization
sim_train_aug = 0.52 + 0.30 * (1 - np.exp(-sim_epochs / 4.5))
sim_val_aug = 0.52 + 0.27 * (1 - np.exp(-sim_epochs / 4.0))

diag2.plot(sim_epochs, sim_train_aug, 'g-', linewidth=2.5, label='Training Accuracy (Learning Invariant Features)')
diag2.plot(sim_epochs, sim_val_aug, 'r--', linewidth=2.5, label='Validation Accuracy (Tracking Training)')
diag2.fill_between(sim_epochs, sim_train_aug, sim_val_aug, color='lightgreen', alpha=0.25, label='Controlled Generalization Gap')
diag2.set_title('Conceptual Model: With Data Augmentation\n(Forced Learning of Robust Generalizable Features)', fontsize=12, fontweight='bold')
diag2.set_xlabel('Epochs', fontsize=11)
diag2.set_ylabel('Accuracy', fontsize=11)
diag2.set_ylim([0.4, 1.02])
diag2.legend(loc='lower right')
diag2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()


## 15. Final Conclusion

### University Presentation Summary & Key Takeaways

1. **Dataset Integrity Comes First:**
   * In raw real-world datasets, corrupted or malformed files are common. Running `Image.verify()` with PIL is essential before launching heavy deep learning workflows.
2. **Small Datasets Suffer Severe Overfitting:**
   * When training on only 2,000 images, unregularized CNNs quickly achieve high training accuracy (~85%–95%) while validation accuracy stalls early (~70%–75%), demonstrating an expanding generalization gap.
3. **Data Augmentation is an Essential Regularizer:**
   * `ImageDataGenerator` dynamically applies transformations (rotation, shifting, zooming, horizontal flipping) in memory on-the-fly with zero additional storage cost.
   * This breaks sample memorization and forces the network to learn invariant geometric features, keeping training and validation performance coupled.
4. **The Validation Rule is Absolute:**
   * Validation and test sets must **never** be augmented. They must reflect the true, unperturbed distribution of unseen data.
5. **Next Steps for Higher Accuracy:**
   * To push accuracy beyond 85%–95% on small datasets, François Chollet recommends combining Data Augmentation with:
     * **Dropout Regularization:** Adding `layers.Dropout(0.5)` before the dense classification layer.
     * **Transfer Learning:** Using pretrained feature extractors (e.g., VGG16, MobileNetV2, ResNet50) trained on ImageNet and fine-tuning top layers.

---
**Repository:** [github.com/SayedaHaneen/data-augmentation-imagedatagenerator](https://github.com/SayedaHaneen/data-augmentation-imagedatagenerator)  
**Notebook:** `Data_Augmentation_ImageDataGenerator.ipynb`
